In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install node2vec tensorflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 114.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 50.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.1
    Uninstalling scipy-1.16.1:
      Successfully uninstalled scipy-1.16.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requ

In [ ]:
 !pip uninstall -y jax numpy
 !pip cache purge

Found existing installation: jax 0.5.3
Uninstalling jax-0.5.3:
  Successfully uninstalled jax-0.5.3
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Files removed: 24


In [ ]:
 !pip install numpy==1.23.5
 !pip install --upgrade jax jaxlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 97.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
orbax-checkpoint 0.11.21 requires jax>=0.5.0, which is not installed.
optax 0.2.5 requires jax>=0.4.27, which is not installed.
dopamine-rl 4.1.2 requires jax>=0.1.72, which is not installed.
flax 0.10.6 requires jax>=0.5.1, which is not installed.
chex 0.1.90 requires jax>=0.4.27, which is not installed.
node2vec 0.5.0 requires numpy<2.0.0,>=1.24.0, but you have numpy 1.23.5 which is incompatible.
imbalanced-learn 0.13.0 requires numpy<3,>=1.24.3, but you have numpy 1.23.5 which is incompatible.
treescope 0.1.10 requires numpy>=1.25.2, but you have numpy 1.23.5 which is incompatible.
scikit-image 0.25.2 requires numpy>=1.24, but you have numpy 1.23.5 which is incompatible.
pymc 5.25.1 requires numpy>=1.25.0, but you have numpy 1.23.5 w

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 104.1 MB/s eta 0:00:00
^C


In [111]:
import pandas as pd
from tqdm import tqdm
import json
import os
import random
import math
import pickle
import numpy as np
import scipy.sparse as sp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, confusion_matrix
from collections import deque

import networkx as nx
import warnings
import keras
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras import activations, initializers, constraints, regularizers
from tensorflow.keras.layers import Input, Layer, Lambda, Dropout, Reshape, Dense, Embedding, LeakyReLU, Maximum
from tensorflow.keras.callbacks import EarlyStopping

from tensorflow.keras import layers, optimizers, losses, metrics, Model
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure magic command is only used in an interactive environment
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except NameError:
    pass


In [112]:
dataFrame = pd.read_csv("/content/drive/MyDrive/Journal/citation_sentiment_corpus.txt", sep = "	", header = None)
dataFrame.columns = ["Source_PaperID", "Target_PaperID", "Sentiment", "Citation_text"]
dataFrame.Sentiment = dataFrame.Sentiment.replace({"o": 1,"p": 2,"n": 0})

/tmp/ipython-input-2858447318.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataFrame.Sentiment = dataFrame.Sentiment.replace({"o": 1,"p": 2,"n": 0})


In [113]:
dataFrame.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8736 entries, 0 to 8735
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Source_PaperID  8736 non-null   object
 1   Target_PaperID  8736 non-null   object
 2   Sentiment       8736 non-null   int64 
 3   Citation_text   8736 non-null   object
dtypes: int64(1), object(3)
memory usage: 273.1+ KB


In [114]:
Source = dataFrame['Source_PaperID']
Target = dataFrame['Target_PaperID']
Sentiment = dataFrame['Sentiment']

In [115]:
import math

def directed_preferential_attachment(graph, edges):
    """
    Preferential Attachment for directed graphs:
    PA(u, v) = out_degree(u) * in_degree(v)
    """
    scores = []
    for u, v in edges:
        score = graph.out_degree(u) * graph.in_degree(v)
        scores.append((u, v, score))
    return scores


def directed_resource_allocation_index(graph, edges):
    """
    Resource Allocation Index for directed graphs:
    Use common nodes that are successors of u and predecessors of v.
    RA(u, v) = sum over w ∈ Γ_out(u) ∩ Γ_in(v) of 1 / degree(w)
    """
    scores = []
    for u, v in edges:
        common_neighbors = set(graph.successors(u)).intersection(set(graph.predecessors(v)))
        score = sum(1 / graph.degree(w) for w in common_neighbors if graph.degree(w) > 0)
        scores.append((u, v, score))
    return scores


def directed_jaccard_coefficient(graph, edges):
    """
    Jaccard Coefficient for directed graphs:
    JC(u, v) = |Γ_out(u) ∩ Γ_in(v)| / |Γ_out(u) ∪ Γ_in(v)|
    """
    scores = []
    for u, v in edges:
        succ_u = set(graph.successors(u))     # out-neighbors of u
        pred_v = set(graph.predecessors(v))   # in-neighbors of v
        intersection = succ_u & pred_v
        union = succ_u | pred_v
        score = len(intersection) / len(union) if union else 0
        scores.append((u, v, score))
    return scores


def directed_adamic_adar_index(graph, edges):
    """
    Adamic-Adar Index for directed graphs:
    AA(u, v) = sum over w ∈ Γ_out(u) ∩ Γ_in(v) of 1 / log(degree(w))
    """
    scores = []
    for u, v in edges:
        succ_u = set(graph.successors(u))     # out-neighbors of u
        pred_v = set(graph.predecessors(v))   # in-neighbors of v
        common_neighbors = succ_u & pred_v
        score = sum(1 / math.log(graph.degree(w)) for w in common_neighbors if graph.degree(w) > 1)
        scores.append((u, v, score))
    return scores


In [116]:
import networkx as nx
import pandas as pd


# Create a DataFrame with source, target, and optional weights
edges = pd.DataFrame({
    "source": Source,
    "target": Target,
    "weight": [1] * len(Source)  # or use real weights if available
})

# Build a directed graph from the DataFrame
G = nx.from_pandas_edgelist(
    edges,
    source='source',
    target='target',
    edge_attr=True,  # includes all other columns (like 'weight') as edge attributes
    create_using=nx.DiGraph()
)


In [117]:
# Create a leaderboard for nodes based on their degree (number of connections)
leaderboard = {node: G.degree(node) for node in G.nodes()}

# Convert the leaderboard to a pandas Series and then to a DataFrame
s = pd.Series(leaderboard, name="Citations")
citation_counts = s.to_frame().sort_values("Citations", ascending=False)

# Display the count of unique citation values
citation_value_counts = citation_counts.value_counts()

# Display results
print("Citation Counts:")
print(citation_counts.head())  # Top-ranked nodes by citations
print("\nValue Counts of Citations:")
print(citation_value_counts)


Citation Counts:
          Citations
J93-2004        436
J93-2003        368
P02-1040        305
P03-1021        281
N03-1017        240

Value Counts of Citations:
Citations
1            1755
2             684
3             283
4             129
5              77
6              26
7              14
8              13
9              11
10              7
11              5
15              4
17              4
20              4
14              4
22              3
13              2
18              2
71              2
27              2
30              2
12              2
151             1
101             1
109             1
117             1
121             1
125             1
138             1
172             1
152             1
368             1
182             1
212             1
240             1
281             1
305             1
95              1
31              1
67              1
29              1
16              1
19              1
23              1
24              1
25             

In [118]:
print("Number of Nodes: ",G.number_of_nodes())
print("Number of Edges: ",G.number_of_edges())

Number of Nodes:  3069
Number of Edges:  5042


In [119]:
#!pip install node2vec

In [120]:
df=dataFrame
import pandas as pd
import numpy as np
from node2vec import Node2Vec
import tensorflow_hub as hub
import tensorflow as tf


In [121]:
leaderboard = {}
for x in G.nodes:
 leaderboard[x] = len(G[x])
s = pd.Series(leaderboard, name='Citations')
citation_counts = s.to_frame().sort_values('Citations', ascending=False)
citation_counts.value_counts()

,count
Citations,
1,1785
2,703
3,295
4,121
0,77
5,59
6,19
7,7
8,3


In [122]:
citation_counts = citation_counts.reset_index(level=0)
citation_counts.columns = ['Node', 'Citations']
citation_counts.head()

,Node,Citations
0,W08-0306,8
1,N09-1058,8
2,D07-1070,8
3,N09-1049,7
4,P08-1068,7


In [123]:
print("Number of Nodes: ",G.number_of_nodes())
print("Number of Edges: ",G.number_of_edges())

Number of Nodes:  3069
Number of Edges:  5042


In [124]:
zero_list = []
for i,j in zip(citation_counts['Node'], citation_counts['Citations']):
    if(j == 0):

        zero_list.append(i)
G.remove_nodes_from(zero_list)

In [125]:

# ----- NODE-LEVEL FEATURES -----

# In-degree, Out-degree, Total Degree Centrality
in_degrees = dict(G.in_degree())
out_degrees = dict(G.out_degree())
total_degrees = {node: in_degrees[node] + out_degrees[node] for node in G.nodes()}

# Betweenness, Closeness, Eigenvector Centrality
betweenness = nx.betweenness_centrality(G)
closeness = nx.closeness_centrality(G)
eigenvector_centrality = nx.eigenvector_centrality(G)

# Combine Node Features into a DataFrame
node_features = pd.DataFrame({
    "node": list(G.nodes()),
    "in_degree": [in_degrees[node] for node in G.nodes()],
    "out_degree": [out_degrees[node] for node in G.nodes()],
    "total_degree": [total_degrees[node] for node in G.nodes()],
    "betweenness": [betweenness[node] for node in G.nodes()],
    "closeness": [closeness[node] for node in G.nodes()],
    "eigenvector_centrality": [eigenvector_centrality[node] for node in G.nodes()],
})


# ----- EDGE-LEVEL FEATURES USING CUSTOM FUNCTIONS -----
# Preferential Attachment
pref_attach = directed_preferential_attachment(G, G.edges())

# Resource Allocation Index
resource_allocation_scores = directed_resource_allocation_index(G, G.edges())

# Jaccard Coefficient
jaccard_scores = directed_jaccard_coefficient(G, G.edges())

# Adamic-Adar Index
adamic_adar_scores = directed_adamic_adar_index(G, G.edges())

# Combine Edge Features into a DataFrame
edge_features = pd.DataFrame({
    "source": [u for u, v, _ in pref_attach],
    "target": [v for u, v, _ in pref_attach],
    "preferential_attachment": [score for _, _, score in pref_attach],
    "resource_allocation": [score for _, _, score in resource_allocation_scores],
    "jaccard": [score for _, _, score in jaccard_scores],
    "adamic_adar": [score for _, _, score in adamic_adar_scores],
})



In [126]:
print(edge_features.describe())

       preferential_attachment  resource_allocation      jaccard  adamic_adar
count              1589.000000          1589.000000  1589.000000  1589.000000
mean                141.224670             0.011998     0.002853     0.053190
std                 177.346121             0.050893     0.015371     0.179574
min                   1.000000             0.000000     0.000000     0.000000
25%                  20.000000             0.000000     0.000000     0.000000
50%                  68.000000             0.000000     0.000000     0.000000
75%                 169.000000             0.000000     0.000000     0.000000
max                1185.000000             0.666667     0.250000     1.820478


In [127]:
#further Improvement and Normalization

path_lengths = []
for u, v in G.edges():
    try:
        path_length = nx.shortest_path_length(G, u, v)
    except nx.NetworkXNoPath:
        path_length = -1  # or a high value
    path_lengths.append(path_length)

edge_features["shortest_path_length"] = path_lengths


In [128]:
node_features.shape , edge_features.shape

((2992, 7), (1589, 7))

In [129]:
node_features.columns, edge_features.columns

(Index(['node', 'in_degree', 'out_degree', 'total_degree', 'betweenness',
        'closeness', 'eigenvector_centrality'],
       dtype='object'),
 Index(['source', 'target', 'preferential_attachment', 'resource_allocation',
        'jaccard', 'adamic_adar', 'shortest_path_length'],
       dtype='object'))

In [130]:
print("Number of Nodes: ",G.number_of_nodes())
print("Number of Edges: ",G.number_of_edges())

Number of Nodes:  2992
Number of Edges:  1589


In [131]:
# Graph G must be defined already, e.g., a NetworkX graph

# Parameters for Node2Vec with Restart
p = 0.5  # Probability to return to the previous node
q = 0.4  # Probability to explore outward nodes

# Initialize Node2Vec with parameters
node2vec = Node2Vec(
    G,
    dimensions=128,
    walk_length=80, #length of each random walk
    num_walks=10, # number of random walk per node
    workers=4, # No. of Cores for parallel processing
    p=p,  # Return hyperparameter  Probability to return to the previous node
    q=q,  # In-out hyperparameter  Probability to explore outward nodes
)

# Fit Node2Vec model
node2vec_model = node2vec.fit(window=10, min_count=1, batch_words=4)

# Create a DataFrame for Node2Vec embeddings
node2vec_embeddings = pd.DataFrame(
    [node2vec_model.wv[str(node)] for node in G.nodes()],
    index=[str(node) for node in G.nodes()],
    columns=[f"node2vec_dim_{i}" for i in range(128)]
)

Computing transition probabilities:   0%|          | 0/2992 [00:00<?, ?it/s]

In [132]:
node2vec_embeddings.head()

,node2vec_dim_0,node2vec_dim_1,node2vec_dim_2,node2vec_dim_3,node2vec_dim_4,node2vec_dim_5,node2vec_dim_6,node2vec_dim_7,node2vec_dim_8,node2vec_dim_9,...,node2vec_dim_118,node2vec_dim_119,node2vec_dim_120,node2vec_dim_121,node2vec_dim_122,node2vec_dim_123,node2vec_dim_124,node2vec_dim_125,node2vec_dim_126,node2vec_dim_127
A00-1043,-0.000184,-0.007147,-0.000183,-0.007226,0.003267,0.001609,-0.007370,-0.004543,-0.004035,-0.001062,...,0.002891,0.002471,-0.000540,0.004575,-0.007189,-0.006622,0.002322,0.003705,-0.006849,-0.005528
H05-1033,-0.007657,0.004683,0.002683,-0.001768,0.004127,-0.001697,0.006700,-0.001718,0.006334,0.006903,...,-0.006657,0.006318,0.006884,-0.005249,0.000679,0.003886,0.001286,-0.000036,-0.003363,-0.007653
I05-2009,-0.006188,0.006929,-0.003185,0.001446,0.002066,0.004205,0.007770,-0.004726,0.005959,-0.000106,...,0.001648,-0.000466,0.000572,-0.007737,0.000543,0.006970,0.002559,-0.000825,-0.002028,0.003419
I08-1016,-0.002206,-0.001900,-0.000415,-0.000443,-0.003539,-0.006254,-0.000736,0.000863,0.001010,-0.007577,...,-0.004677,0.003948,0.005782,0.000427,-0.005950,0.004984,-0.003219,0.002630,-0.004612,-0.003487
I08-2101,0.003836,0.006145,0.001279,-0.002051,-0.000024,-0.007271,0.000525,-0.003916,-0.000801,0.006305,...,-0.000479,-0.003645,0.001854,0.007791,-0.007005,0.002254,0.003848,0.002205,-0.007239,0.005193


In [133]:
# Ensure Node2Vec embeddings cover all nodes in the original DataFrame
all_nodes = df["Source_PaperID"].unique()
missing_nodes = set(all_nodes) - set(node2vec_embeddings.index)
for node in missing_nodes:
    node2vec_embeddings.loc[node] = np.zeros(128)

In [134]:
# Reindex to ensure the order matches `all_nodes`
node2vec_embeddings = node2vec_embeddings.reindex(all_nodes)

In [135]:
node2vec_embeddings.shape

(2992, 128)

In [136]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import tensorflow_hub as hub
import tensorflow as tf

# Node2Vec embeddings with `Source_PaperID` as the index
node2vec_embeddings.index.name = "Source_PaperID"

In [137]:

# ----- STEP 2: EXTRACT CITATION TEXTS FOR UNIQUE NODES -----
# Select only rows where `Source_PaperID` matches `node2vec_embeddings` index
citation_texts = df[df["Source_PaperID"].isin(node2vec_embeddings.index)]
citation_texts.shape

(8736, 4)

In [138]:
# Ensure unique rows for `Source_PaperID` (if duplicates exist, group by and take the first)
unique_citation_texts = citation_texts.groupby("Source_PaperID").first().reset_index()

In [139]:
unique_citation_texts.shape

(2992, 4)

In [140]:

combined_embeddings = node2vec_embeddings

# ----- STEP 5: ADD SENTIMENT LABELS -----
# Map sentiment labels from the original DataFrame
sentiment_dict = df.set_index("Source_PaperID")["Sentiment"].to_dict()
combined_embeddings["sentiment"] = combined_embeddings.index.map(sentiment_dict)

# ----- STEP 6: FINAL DATAFRAME -----
# Reset index for final output
final_df = combined_embeddings.reset_index()

# Final DataFrame includes:
# - `Source_PaperID`: Node ID.
# - `node2vec_dim_*`: Node2Vec embeddings (128 dimensions).
# - `sentiment`: Sentiment label (target value).
print(final_df.head())
print("\nShape of the final DataFrame:", final_df.shape)


  Source_PaperID  node2vec_dim_0  node2vec_dim_1  node2vec_dim_2  \
0       A00-1043       -0.000184       -0.007147       -0.000183   
1       H05-1033       -0.007657        0.004683        0.002683   
2       I05-2009       -0.006188        0.006929       -0.003185   
3       I08-1016       -0.002206       -0.001900       -0.000415   
4       I08-2101        0.003836        0.006145        0.001279   

   node2vec_dim_3  node2vec_dim_4  node2vec_dim_5  node2vec_dim_6  \
0       -0.007226        0.003267        0.001609       -0.007370   
1       -0.001768        0.004127       -0.001697        0.006700   
2        0.001446        0.002066        0.004205        0.007770   
3       -0.000443       -0.003539       -0.006254       -0.000736   
4       -0.002051       -0.000024       -0.007271        0.000525   

   node2vec_dim_7  node2vec_dim_8  ...  node2vec_dim_119  node2vec_dim_120  \
0       -0.004543       -0.004035  ...          0.002471         -0.000540   
1       -0.001718   

In [141]:
final_df.head()

,Source_PaperID,node2vec_dim_0,node2vec_dim_1,node2vec_dim_2,node2vec_dim_3,node2vec_dim_4,node2vec_dim_5,node2vec_dim_6,node2vec_dim_7,node2vec_dim_8,...,node2vec_dim_119,node2vec_dim_120,node2vec_dim_121,node2vec_dim_122,node2vec_dim_123,node2vec_dim_124,node2vec_dim_125,node2vec_dim_126,node2vec_dim_127,sentiment
0,A00-1043,-0.000184,-0.007147,-0.000183,-0.007226,0.003267,0.001609,-0.007370,-0.004543,-0.004035,...,0.002471,-0.000540,0.004575,-0.007189,-0.006622,0.002322,0.003705,-0.006849,-0.005528,1
1,H05-1033,-0.007657,0.004683,0.002683,-0.001768,0.004127,-0.001697,0.006700,-0.001718,0.006334,...,0.006318,0.006884,-0.005249,0.000679,0.003886,0.001286,-0.000036,-0.003363,-0.007653,1
2,I05-2009,-0.006188,0.006929,-0.003185,0.001446,0.002066,0.004205,0.007770,-0.004726,0.005959,...,-0.000466,0.000572,-0.007737,0.000543,0.006970,0.002559,-0.000825,-0.002028,0.003419,1
3,I08-1016,-0.002206,-0.001900,-0.000415,-0.000443,-0.003539,-0.006254,-0.000736,0.000863,0.001010,...,0.003948,0.005782,0.000427,-0.005950,0.004984,-0.003219,0.002630,-0.004612,-0.003487,1
4,I08-2101,0.003836,0.006145,0.001279,-0.002051,-0.000024,-0.007271,0.000525,-0.003916,-0.000801,...,-0.003645,0.001854,0.007791,-0.007005,0.002254,0.003848,0.002205,-0.007239,0.005193,2


In [142]:
dataset = final_df

In [143]:
dataset.rename(columns={"Source_PaperID": "Node_id"}, inplace=True)
dataset.rename(columns={"sentiment": "label"}, inplace=True)

In [144]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2992 entries, 0 to 2991
Columns: 130 entries, Node_id to label
dtypes: float32(128), int64(1), object(1)
memory usage: 1.5+ MB


In [145]:
node2vec_cc = dataset

In [146]:
graph = nx.to_pandas_edgelist(G)

graph

,source,target,weight
0,W09-0604,N03-1003,1
1,A00-1031,W96-0213,1
2,A97-1004,W96-0213,1
3,C08-1026,P08-1085,1
4,N01-1023,W96-0213,1
...,...,...,...
1584,W07-1209,W96-0213,1
1585,W07-1516,W96-0213,1
1586,W07-2053,W96-0213,1
1587,W08-0611,W96-0213,1


In [147]:
node2vec_cc["Node_id"] = dataset["Node_id"]
node2vec_cc["label"] = dataset["label"]

In [148]:
paper_idx = {name: idx for idx, name in enumerate(node2vec_cc["Node_id"])}

node2vec_cc["Node_id"] = node2vec_cc["Node_id"].apply(lambda name: paper_idx[name])
graph["source"] = graph["source"].apply(lambda name: paper_idx[name])
graph["target"] = graph["target"].apply(lambda name: paper_idx[name])

In [149]:
node2vec_cc.columns

Index(['Node_id', 'node2vec_dim_0', 'node2vec_dim_1', 'node2vec_dim_2',
       'node2vec_dim_3', 'node2vec_dim_4', 'node2vec_dim_5', 'node2vec_dim_6',
       'node2vec_dim_7', 'node2vec_dim_8',
       ...
       'node2vec_dim_119', 'node2vec_dim_120', 'node2vec_dim_121',
       'node2vec_dim_122', 'node2vec_dim_123', 'node2vec_dim_124',
       'node2vec_dim_125', 'node2vec_dim_126', 'node2vec_dim_127', 'label'],
      dtype='object', length=130)

In [150]:
node2vec_cc.head()

,Node_id,node2vec_dim_0,node2vec_dim_1,node2vec_dim_2,node2vec_dim_3,node2vec_dim_4,node2vec_dim_5,node2vec_dim_6,node2vec_dim_7,node2vec_dim_8,...,node2vec_dim_119,node2vec_dim_120,node2vec_dim_121,node2vec_dim_122,node2vec_dim_123,node2vec_dim_124,node2vec_dim_125,node2vec_dim_126,node2vec_dim_127,label
0,0,-0.000184,-0.007147,-0.000183,-0.007226,0.003267,0.001609,-0.007370,-0.004543,-0.004035,...,0.002471,-0.000540,0.004575,-0.007189,-0.006622,0.002322,0.003705,-0.006849,-0.005528,1
1,1,-0.007657,0.004683,0.002683,-0.001768,0.004127,-0.001697,0.006700,-0.001718,0.006334,...,0.006318,0.006884,-0.005249,0.000679,0.003886,0.001286,-0.000036,-0.003363,-0.007653,1
2,2,-0.006188,0.006929,-0.003185,0.001446,0.002066,0.004205,0.007770,-0.004726,0.005959,...,-0.000466,0.000572,-0.007737,0.000543,0.006970,0.002559,-0.000825,-0.002028,0.003419,1
3,3,-0.002206,-0.001900,-0.000415,-0.000443,-0.003539,-0.006254,-0.000736,0.000863,0.001010,...,0.003948,0.005782,0.000427,-0.005950,0.004984,-0.003219,0.002630,-0.004612,-0.003487,1
4,4,0.003836,0.006145,0.001279,-0.002051,-0.000024,-0.007271,0.000525,-0.003916,-0.000801,...,-0.003645,0.001854,0.007791,-0.007005,0.002254,0.003848,0.002205,-0.007239,0.005193,2


In [151]:
df=node2vec_cc

In [152]:
node2vec_cc

,Node_id,node2vec_dim_0,node2vec_dim_1,node2vec_dim_2,node2vec_dim_3,node2vec_dim_4,node2vec_dim_5,node2vec_dim_6,node2vec_dim_7,node2vec_dim_8,...,node2vec_dim_119,node2vec_dim_120,node2vec_dim_121,node2vec_dim_122,node2vec_dim_123,node2vec_dim_124,node2vec_dim_125,node2vec_dim_126,node2vec_dim_127,label
0,0,-0.000184,-0.007147,-0.000183,-0.007226,0.003267,0.001609,-0.007370,-0.004543,-0.004035,...,0.002471,-0.000540,0.004575,-0.007189,-0.006622,0.002322,0.003705,-0.006849,-0.005528,1
1,1,-0.007657,0.004683,0.002683,-0.001768,0.004127,-0.001697,0.006700,-0.001718,0.006334,...,0.006318,0.006884,-0.005249,0.000679,0.003886,0.001286,-0.000036,-0.003363,-0.007653,1
2,2,-0.006188,0.006929,-0.003185,0.001446,0.002066,0.004205,0.007770,-0.004726,0.005959,...,-0.000466,0.000572,-0.007737,0.000543,0.006970,0.002559,-0.000825,-0.002028,0.003419,1
3,3,-0.002206,-0.001900,-0.000415,-0.000443,-0.003539,-0.006254,-0.000736,0.000863,0.001010,...,0.003948,0.005782,0.000427,-0.005950,0.004984,-0.003219,0.002630,-0.004612,-0.003487,1
4,4,0.003836,0.006145,0.001279,-0.002051,-0.000024,-0.007271,0.000525,-0.003916,-0.000801,...,-0.003645,0.001854,0.007791,-0.007005,0.002254,0.003848,0.002205,-0.007239,0.005193,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2987,2987,0.001841,0.003810,-0.006684,-0.004222,0.005398,0.005875,0.000888,0.001820,-0.003048,...,0.002968,-0.003288,-0.002919,-0.005430,-0.007413,0.003467,0.004787,-0.001710,0.001443,1
2988,2988,-0.003076,-0.003147,0.002801,-0.002224,0.005725,-0.001184,-0.004182,0.002927,-0.007018,...,-0.003416,0.001687,0.005710,-0.002658,-0.006115,0.008836,0.003029,0.008690,-0.004852,2
2989,2989,0.001856,-0.002206,-0.005617,0.005000,-0.003527,0.004461,0.003209,0.006328,-0.002641,...,0.003512,0.002455,0.005590,0.007174,-0.006718,0.005271,0.006028,0.003203,-0.002054,1
2990,2990,0.008187,-0.002447,-0.004999,-0.008528,-0.005955,-0.003181,0.001875,-0.004151,0.006300,...,0.000940,0.006603,-0.001809,-0.004739,0.005644,0.004139,0.004943,0.006246,0.001437,1


In [153]:
df = df.drop(columns=["Node_id"])

In [154]:
# Features and labels
X = df.drop(columns=["label"]).values
y = df["label"].values

In [155]:
# ---- First split: 70% train, 30% temp (val+test) ----
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# ---- Second split: from temp, split into 60% val and 40% test ----
# (because 0.60 * 0.30 = 0.18 and 0.40 * 0.30 = 0.12)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.40, stratify=y_temp, random_state=42
)

#Scores

In [156]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# create a logistic regression model
model = LogisticRegression()

# fit the model to the training data
model.fit(X_train, y_train)

# make predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)
y_val_pred = model.predict(X_val)

# evaluate accuracy
Lr_test_accuracy = accuracy_score(y_test, y_test_pred)
Lr_val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"Testing accuracy: {Lr_test_accuracy:.4f}")
print(f"Validation accuracy: {Lr_val_accuracy:.4f}")

# macro, micro, and weighted metrics for test set
for avg in ['macro', 'micro', 'weighted']:
    print(f"\n--- {avg.upper()} ---")
    print(f"Precision: {precision_score(y_test, y_test_pred, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_test, y_test_pred, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_test, y_test_pred, average=avg):.4f}")


Testing accuracy: 0.8778
Validation accuracy: 0.8755

--- MACRO ---
Precision: 0.2926
Recall:    0.3333
F1-score:  0.3116

--- MICRO ---
Precision: 0.8778
Recall:    0.8778
F1-score:  0.8778

--- WEIGHTED ---
Precision: 0.7705
Recall:    0.8778
F1-score:  0.8206


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [157]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Define the parameter grid to search
param_grid = {
    'max_depth': [5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Initialize the Decision Tree classifier
dt_model = DecisionTreeClassifier()

# Use GridSearchCV to find the best hyperparameters
grid_search = GridSearchCV(estimator=dt_model, param_grid=param_grid, cv=5, scoring='accuracy', verbose=1, n_jobs=-1)

# Fit GridSearchCV to the training data
grid_search.fit(X_train, y_train)

# Get the best parameters and best estimator
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

print("Best parameters found: ", best_params)

# Evaluate on validation data
y_pred_val = best_model.predict(X_val)
Dt_val_accuracy = accuracy_score(y_val, y_pred_val)
print(f"\nValidation accuracy: {Dt_val_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nValidation ({avg.upper()}):")
    print(f"Precision: {precision_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_val, y_pred_val, average=avg):.4f}")

# Evaluate on test data
y_pred_test = best_model.predict(X_test)
Dt_test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"\nTesting accuracy: {Dt_test_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nTesting ({avg.upper()}):")
    print(f"Precision: {precision_score(y_test, y_pred_test, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred_test, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_test, y_pred_test, average=avg):.4f}")



Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters found:  {'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 2}

Validation accuracy: 0.8439

Validation (MACRO):
Precision: 0.3085
Recall:    0.3269
F1-score:  0.3143

Validation (MICRO):
Precision: 0.8439
Recall:    0.8439
F1-score:  0.8439

Validation (WEIGHTED):
Precision: 0.7693
Recall:    0.8439
F1-score:  0.8039

Testing accuracy: 0.8639

Testing (MACRO):
Precision: 0.3768
Recall:    0.3450
F1-score:  0.3399

Testing (MICRO):
Precision: 0.8639
Recall:    0.8639
F1-score:  0.8639

Testing (WEIGHTED):
Precision: 0.7971
Recall:    0.8639
F1-score:  0.8223


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [158]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Define the parameter grid to search
param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'gamma': ['scale', 'auto']
}

# Initialize the SVM classifier
svm_model = SVC()

# Use GridSearchCV to find the best hyperparameters
grid_search = GridSearchCV(estimator=svm_model, param_grid=param_grid, cv=5,
                           scoring='accuracy', verbose=1, n_jobs=-1)

# Fit GridSearchCV to the training data
grid_search.fit(X_train, y_train)

# Get the best parameters and best estimator
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

print("Best parameters found: ", best_params)

# ---------- Validation evaluation ----------
y_pred_val = best_model.predict(X_val)
Svm_val_accuracy = accuracy_score(y_val, y_pred_val)
print(f"\nValidation accuracy: {Svm_val_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nValidation ({avg.upper()}):")
    print(f"Precision: {precision_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_val, y_pred_val, average=avg):.4f}")

# ---------- Testing evaluation ----------
y_pred_test = best_model.predict(X_test)
Svm_test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"\nTesting accuracy: {Svm_test_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nTesting ({avg.upper()}):")
    print(f"Precision: {precision_score(y_test, y_pred_test, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred_test, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_test, y_pred_test, average=avg):.4f}")



Fitting 5 folds for each of 32 candidates, totalling 160 fits
Best parameters found:  {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}

Validation accuracy: 0.8755

Validation (MACRO):
Precision: 0.2918
Recall:    0.3333
F1-score:  0.3112

Validation (MICRO):
Precision: 0.8755
Recall:    0.8755
F1-score:  0.8755

Validation (WEIGHTED):
Precision: 0.7664
Recall:    0.8755
F1-score:  0.8173

Testing accuracy: 0.8778

Testing (MACRO):
Precision: 0.2926
Recall:    0.3333
F1-score:  0.3116

Testing (MICRO):
Precision: 0.8778
Recall:    0.8778
F1-score:  0.8778

Testing (WEIGHTED):
Precision: 0.7705
Recall:    0.8778
F1-score:  0.8206


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/m

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Define the parameter grid to search
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

# Initialize the Extra Trees classifier
et_model = ExtraTreesClassifier(random_state=42)

# Use GridSearchCV to find the best hyperparameters
grid_search = GridSearchCV(estimator=et_model, param_grid=param_grid, cv=5,
                           scoring='accuracy', verbose=1, n_jobs=-1)

# Fit GridSearchCV to the training data
grid_search.fit(X_train, y_train)

# Get the best parameters and the best estimator
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

print("Best parameters found: ", best_params)

# ---- Function to print metrics for any split ----
def evaluate_model(name, y_true, y_pred):
    print(f"\n{name} metrics (GridSearchCV):")
    print("Accuracy: {:.2f}%".format(accuracy_score(y_true, y_pred) * 100))

    # Weighted
    print("Precision (weighted): {:.4f}".format(precision_score(y_true, y_pred, average='weighted')))
    print("Recall (weighted): {:.4f}".format(recall_score(y_true, y_pred, average='weighted')))
    print("F1 Score (weighted): {:.4f}".format(f1_score(y_true, y_pred, average='weighted')))

    # Macro
    print("Precision (macro): {:.4f}".format(precision_score(y_true, y_pred, average='macro')))
    print("Recall (macro): {:.4f}".format(recall_score(y_true, y_pred, average='macro')))
    print("F1 Score (macro): {:.4f}".format(f1_score(y_true, y_pred, average='macro')))

    # Micro
    print("Precision (micro): {:.4f}".format(precision_score(y_true, y_pred, average='micro')))
    print("Recall (micro): {:.4f}".format(recall_score(y_true, y_pred, average='micro')))
    print("F1 Score (micro): {:.4f}".format(f1_score(y_true, y_pred, average='micro')))

# Evaluate on validation data
evaluate_model("Validation", y_val, best_model.predict(X_val))

# Evaluate on testing data
evaluate_model("Testing", y_test, best_model.predict(X_test))


Fitting 5 folds for each of 216 candidates, totalling 1080 fits


In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Initialize the AdaBoost classifier
n_estimators = 100
learning_rate = 1.0
algorithm = 'SAMME'
model = AdaBoostClassifier(n_estimators=n_estimators,
                           learning_rate=learning_rate,
                           algorithm=algorithm)

# Train the AdaBoost classifier
model.fit(X_train, y_train)

# ---------- Validation evaluation ----------
y_pred_val = model.predict(X_val)
Ada_val_accuracy = accuracy_score(y_val, y_pred_val)
print(f"\nValidation accuracy: {Ada_val_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nValidation ({avg.upper()}):")
    print(f"Precision: {precision_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_val, y_pred_val, average=avg):.4f}")

# ---------- Testing evaluation ----------
y_pred_test = model.predict(X_test)
Ada_test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"\nTesting accuracy: {Ada_test_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nTesting ({avg.upper()}):")
    print(f"Precision: {precision_score(y_test, y_pred_test, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred_test, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_test, y_pred_test, average=avg):.4f}")


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Define the parameter grid to search
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# Initialize the Random Forest classifier
rf_model = RandomForestClassifier(random_state=42)

# Use GridSearchCV to find the best hyperparameters
grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=5, scoring='accuracy', verbose=1, n_jobs=-1)

# Fit GridSearchCV to the training data
grid_search.fit(X_train, y_train)

# Get the best parameters and best estimator
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

print("Best parameters found: ", best_params)

# ---------- Validation evaluation ----------
y_pred_val = best_model.predict(X_val)
Rf_val_accuracy = accuracy_score(y_val, y_pred_val)
print(f"\nValidation accuracy: {Rf_val_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nValidation ({avg.upper()}):")
    print(f"Precision: {precision_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_val, y_pred_val, average=avg):.4f}")

# ---------- Testing evaluation ----------
y_pred_test = best_model.predict(X_test)
Rf_test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"\nTesting accuracy: {Rf_test_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nTesting ({avg.upper()}):")
    print(f"Precision: {precision_score(y_test, y_pred_test, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred_test, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_test, y_pred_test, average=avg):.4f}")


Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Best parameters found:  {'bootstrap': False, 'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 150}

Validation accuracy: 0.8755

Validation (MACRO):
Precision: 0.2918
Recall:    0.3333
F1-score:  0.3112

Validation (MICRO):
Precision: 0.8755
Recall:    0.8755
F1-score:  0.8755

Validation (WEIGHTED):
Precision: 0.7664
Recall:    0.8755
F1-score:  0.8173

Testing accuracy: 0.8778

Testing (MACRO):
Precision: 0.2926
Recall:    0.3333
F1-score:  0.3116

Testing (MICRO):
Precision: 0.8778
Recall:    0.8778
F1-score:  0.8778

Testing (WEIGHTED):
Precision: 0.7705
Recall:    0.8778
F1-score:  0.8206


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/m

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Initialize the SGD classifier
alpha = 0.0001
max_iter = 1000
tol = 1e-3
model = SGDClassifier(alpha=alpha, max_iter=max_iter, tol=tol)

# Train the SGD classifier
model.fit(X_train, y_train)

# ---------- Validation evaluation ----------
y_pred_val = model.predict(X_val)
Sgd_val_accuracy = accuracy_score(y_val, y_pred_val)
print(f"\nValidation accuracy: {Sgd_val_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nValidation ({avg.upper()}):")
    print(f"Precision: {precision_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_val, y_pred_val, average=avg):.4f}")

# ---------- Testing evaluation ----------
y_pred_test = model.predict(X_test)
Sgd_test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"\nTesting accuracy: {Sgd_test_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nTesting ({avg.upper()}):")
    print(f"Precision: {precision_score(y_test, y_pred_test, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred_test, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_test, y_pred_test, average=avg):.4f}")



Validation accuracy: 0.8755

Validation (MACRO):
Precision: 0.2918
Recall:    0.3333
F1-score:  0.3112

Validation (MICRO):
Precision: 0.8755
Recall:    0.8755
F1-score:  0.8755

Validation (WEIGHTED):
Precision: 0.7664
Recall:    0.8755
F1-score:  0.8173

Testing accuracy: 0.8778

Testing (MACRO):
Precision: 0.2926
Recall:    0.3333
F1-score:  0.3116

Testing (MICRO):
Precision: 0.8778
Recall:    0.8778
F1-score:  0.8778

Testing (WEIGHTED):
Precision: 0.7705
Recall:    0.8778
F1-score:  0.8206


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/m

In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss

# Initialize the base models
lr_model = LogisticRegression(max_iter=1000)
# Change the loss to 'log_loss' for probability estimates
sgd_model = SGDClassifier(loss='log_loss', max_iter=1000, tol=1e-3)

# Create the Voting Classifier (change voting='hard' for hard voting)
vc_model = VotingClassifier(
    estimators=[('lr', lr_model), ('sgd', sgd_model)],
    voting='soft'  # use 'hard' for majority voting
)

# Train the voting classifier
vc_model.fit(X_train, y_train)

# ---------- Validation evaluation ----------
y_pred_val = vc_model.predict(X_val)
vc_val_accuracy = accuracy_score(y_val, y_pred_val)
print(f"\nValidation accuracy: {vc_val_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nValidation ({avg.upper()}):")
    print(f"Precision: {precision_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_val, y_pred_val, average=avg):.4f}")

# ---------- Testing evaluation ----------
y_pred_test = vc_model.predict(X_test)
vc_test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"\nTesting accuracy: {vc_test_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nTesting ({avg.upper()}):")
    print(f"Precision: {precision_score(y_test, y_test_pred, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_test, y_test_pred, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_test, y_test_pred, average=avg):.4f}")


Validation accuracy: 0.8755

Validation (MACRO):
Precision: 0.2918
Recall:    0.3333
F1-score:  0.3112

Validation (MICRO):
Precision: 0.8755
Recall:    0.8755
F1-score:  0.8755

Validation (WEIGHTED):
Precision: 0.7664
Recall:    0.8755
F1-score:  0.8173

Testing accuracy: 0.8778

Testing (MACRO):
Precision: 0.2926
Recall:    0.3333
F1-score:  0.3116

Testing (MICRO):
Precision: 0.8778
Recall:    0.8778
F1-score:  0.8778

Testing (WEIGHTED):
Precision: 0.7705
Recall:    0.8778
F1-score:  0.8206


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/m